In [1]:
"""
02_eda.py
---------
Exploratory data analysis on the synthetic CLV dataset.

Reads:
    data/raw/customers.csv
    data/raw/transactions.csv

Produces:
    - printed summary statistics
    - eda_outputs/*.png  (distribution and trend plots)

Goal: confirm the dataset behaves sensibly before building any
features or models, and spot anything (outliers, missing history,
etc.) that needs handling in the next phase.
"""

import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
OUTPUT_DIR = "eda_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ----------------------------------------------------------------
# 1. LOAD DATA
# ----------------------------------------------------------------
customers = pd.read_csv("data/raw/customers.csv", parse_dates=["acquisition_date"])
transactions = pd.read_csv("data/raw/transactions.csv", parse_dates=["transaction_date"])

print(f"Customers: {len(customers)}")
print(f"Transactions: {len(transactions)}")
print(f"Date range of transactions: {transactions['transaction_date'].min().date()} "
      f"to {transactions['transaction_date'].max().date()}\n")

# ----------------------------------------------------------------
# 2. PER-CUSTOMER SUMMARY
# ----------------------------------------------------------------
per_customer = transactions.groupby("customer_id").agg(
    purchase_count=("amount", "count"),
    total_spend=("amount", "sum"),
    avg_order_value=("amount", "mean"),
    first_purchase=("transaction_date", "min"),
    last_purchase=("transaction_date", "max"),
).reset_index()

per_customer["lifespan_days"] = (
    per_customer["last_purchase"] - per_customer["first_purchase"]
).dt.days

per_customer = per_customer.merge(customers, on="customer_id")

# ----------------------------------------------------------------
# 3. SUMMARY STATISTICS
# ----------------------------------------------------------------
print("Purchase count per customer:")
print(per_customer["purchase_count"].describe().round(2), "\n")

print("Average order value per customer:")
print(per_customer["avg_order_value"].describe().round(2), "\n")

print("Customer lifespan in days (first to last purchase):")
print(per_customer["lifespan_days"].describe().round(2), "\n")

one_time_buyers = (per_customer["purchase_count"] == 1).sum()
print(f"One-time buyers (never returned): {one_time_buyers} "
      f"({one_time_buyers / len(per_customer):.1%} of customers)\n")

high_freq_threshold = per_customer["purchase_count"].quantile(0.99)
high_freq_customers = (per_customer["purchase_count"] > high_freq_threshold).sum()
print(f"99th percentile purchase count: {high_freq_threshold:.0f} "
      f"-> {high_freq_customers} customers above this (potential outliers to review)\n")

# ----------------------------------------------------------------
# 4. PLOTS
# ----------------------------------------------------------------

# (a) Purchase frequency distribution
plt.figure(figsize=(7, 4))
sns.histplot(per_customer["purchase_count"], bins=30)
plt.title("Distribution of Purchase Count per Customer")
plt.xlabel("Number of Purchases")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/purchase_frequency_distribution.png", dpi=120)
plt.close()

# (b) Average order value distribution
plt.figure(figsize=(7, 4))
sns.histplot(per_customer["avg_order_value"], bins=30)
plt.title("Distribution of Average Order Value per Customer")
plt.xlabel("Average Order Value")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/order_value_distribution.png", dpi=120)
plt.close()

# (c) Customer lifespan distribution
plt.figure(figsize=(7, 4))
sns.histplot(per_customer["lifespan_days"], bins=30)
plt.title("Distribution of Customer Lifespan (days)")
plt.xlabel("Days Between First and Last Purchase")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/lifespan_distribution.png", dpi=120)
plt.close()

# (d) Acquisition volume over time
monthly_acquisitions = (
    customers.set_index("acquisition_date")
    .resample("ME")
    .size()
)
plt.figure(figsize=(8, 4))
monthly_acquisitions.plot(kind="line", marker="o")
plt.title("Customer Acquisitions per Month")
plt.xlabel("Month")
plt.ylabel("New Customers")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/acquisitions_over_time.png", dpi=120)
plt.close()

# (e) Total spend by channel
spend_by_channel = per_customer.groupby("acquisition_channel")["total_spend"].mean().sort_values(ascending=False)
plt.figure(figsize=(7, 4))
sns.barplot(x=spend_by_channel.index, y=spend_by_channel.values)
plt.title("Average Total Spend per Customer, by Acquisition Channel")
plt.ylabel("Average Total Spend")
plt.xlabel("Acquisition Channel")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/spend_by_channel.png", dpi=120)
plt.close()

print(f"Plots saved to ./{OUTPUT_DIR}/")

Customers: 4000
Transactions: 42241
Date range of transactions: 2022-01-01 to 2024-06-30

Purchase count per customer:
count    4000.00
mean       10.56
std         7.73
min         1.00
25%         4.00
50%        10.00
75%        15.00
max        50.00
Name: purchase_count, dtype: float64 

Average order value per customer:
count    4000.00
mean     1194.16
std       269.27
min       442.48
25%      1023.52
50%      1162.58
75%      1298.83
max      2237.31
Name: avg_order_value, dtype: float64 

Customer lifespan in days (first to last purchase):
count    4000.00
mean      354.46
std       243.65
min         0.00
25%       134.75
50%       358.00
75%       533.00
max       909.00
Name: lifespan_days, dtype: float64 

One-time buyers (never returned): 283 (7.1% of customers)

99th percentile purchase count: 39 -> 35 customers above this (potential outliers to review)

Plots saved to ./eda_outputs/
